# Week 30, Day 1: The voltage clamp: separating sodium and potassium currents

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w030/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Currents under voltage clamp

Write `gate(V, name, t)`, which returns a gating variable's time course after a step from -65 mV to V, using `inf_tau`. Then complete `clamp(V)`, returning the sodium and potassium currents. Run it for steps from -45 to +75 mV.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
gNa, gK, gL, ENa, EK, EL = 120.0, 36.0, 0.3, 50.0, -77.0, -54.387

def rates(V):
    V = V + 1e-7                                     # avoids 0/0 at -55 and -40 mV
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return {"n": (an, bn), "m": (am, bm), "h": (ah, bh)}

def inf_tau(V, name):
    a, b = rates(V)[name]
    return a / (a + b), 1 / (a + b)

t = np.arange(0, 10, 0.01)                           # ms

def gate(V, name, t):
    xinf, tau = inf_tau(V, name)
    x0 = inf_tau(-65.0, name)[0]
    return ...

def clamp(V):
    m, h, n = gate(V, "m", t), gate(V, "h", t), gate(V, "n", t)
    INa = ...
    IK = ...
    return INa, IK

for V in range(-45, 80, 20):
    INa, IK = clamp(V)
    plt.plot(t, INa + IK, label=f"{V} mV")
    print(f"{V:4d} mV: peak sodium {INa[np.argmax(abs(INa))]:7.0f}, potassium at 10 ms {IK[-1]:6.0f} uA/cm^2")
plt.xlabel("ms"); plt.ylabel("total ionic current (uA/cm^2)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_gates():
    "gate returns the exponential relaxation from rest towards the steady value"
    ninf, tau = inf_tau(0.0, "n"); n0 = inf_tau(-65.0, "n")[0]
    assert np.allclose(gate(0.0, "n", t), ninf - (ninf - n0) * np.exp(-t / tau))

def test_currents():
    "at 0 mV the sodium current is inward at first and the potassium current outward and sustained"
    INa, IK = clamp(0.0)
    assert INa.min() < -1000 and abs(INa[-1]) < 100 and IK[-1] > 1500

test_gates()
test_currents()
print("All checks passed")

### Exercise: Subtracting the TTX-resistant current

Two noisy recordings at 0 mV are simulated: the control current and the current after TTX (which leaves only potassium and a small leak, here omitted). Recover the sodium current `I_Na_est` by subtraction, then the conductances `g_Na_est` and `g_K_est` by dividing each current by its driving force at 0 mV.

In [ ]:
r = np.random.default_rng(1952)
INa, IK = clamp(0.0)
I_control = INa + IK + r.normal(0, 15, t.size)
I_ttx = IK + r.normal(0, 15, t.size)

I_Na_est = ...
g_Na_est = ...
g_K_est = ...
print(f"peak g_Na {g_Na_est.max():.1f} mS/cm^2 at {t[np.argmax(g_Na_est)]:.2f} ms; g_K at 10 ms {g_K_est[-1]:.1f} mS/cm^2")
plt.plot(t, g_Na_est, label="g_Na"); plt.plot(t, g_K_est, label="g_K")
plt.xlabel("ms"); plt.ylabel("conductance (mS/cm^2)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_subtract():
    "I_Na_est is the control current minus the current in TTX"
    assert np.allclose(I_Na_est, I_control - I_ttx)

def test_conductance():
    "the conductances divide each current by its driving force, and peak g_Na is near 29 mS/cm^2 at 0 mV"
    assert np.allclose(g_Na_est, I_Na_est / (0.0 - ENa)) and np.allclose(g_K_est, I_ttx / (0.0 - EK))
    assert 27 < g_Na_est.max() < 31

test_subtract()
test_conductance()
print("All checks passed")

### Exercise: The activation curve

For steps from -70 to +45 mV in 5 mV steps (skipping +50, where the driving force is zero), compute the peak sodium conductance `g_peak`. Fit g_max / (1 + exp(-(V - V_half) / k)) by grid search over V_half from -50 to 0 and k from 2 to 20, with g_max fixed at the largest value.

In [ ]:
Vs = np.arange(-70, 46, 5.0)
g_peak = np.array([0.0 for V in Vs])            # replace 0.0 with the peak sodium conductance at V
gmax = g_peak.max()
best = (np.inf, None, None)
for vh in np.arange(-50, 0.01, 0.5):
    for k in np.arange(2, 20.01, 0.25):
        err = ...
        if err < best[0]:
            best = (err, vh, k)
_, V_half, k_slope = best
print(f"half-activation {V_half:.1f} mV, slope factor {k_slope:.2f} mV")
plt.plot(Vs, g_peak, "o"); plt.plot(Vs, gmax / (1 + np.exp(-(Vs - V_half) / k_slope)))
plt.xlabel("step voltage (mV)"); plt.ylabel("peak g_Na (mS/cm^2)"); plt.show()

In [ ]:
import numpy as np
def test_peaks():
    "g_peak holds the peak sodium conductance at each voltage"
    INa, _ = clamp(Vs[5])
    assert abs(g_peak[5] - (INa / (Vs[5] - ENa)).max()) < 1e-9

def test_fit():
    "the Boltzmann fit has its half-activation between -20 and -5 mV and a slope factor between 10 and 20 mV"
    assert -20 < V_half < -5 and 10 < k_slope < 20

test_peaks()
test_fit()
print("All checks passed")